In [ ]:
#use "../lib/nb.ml";;

# Converting <span style="font-variant:small-caps;">Html</span> to Text

This notebook shows how we can use the module `Re`
to extract the text that is embedded in an <span style="font-variant:small-caps;">Html</span> file.  
In order to be concise, it only supports a small subset of 
<span style="font-variant:small-caps;">Html</span>.  Below is the content of my old
<a href="http://wwwlehre.dhbw-stuttgart.de/~stroetma/">web page</a> that I had used when I was still working at the DHBW Stuttgart.  The goal of this notebook is to write 
a sequence of regular expressions to extract the text from this web page.

In [ ]:
let data = {|
<!doctype html>
<html>
  <head>
    <meta charset="utf-8">
    <title>Homepage of Prof. Dr. Karl Stroetmann</title>
    <link type="text/css" rel="stylesheet" href="style.css" />
    <link href="http://fonts.googleapis.com/css?family=Rochester&subset=latin,latin-ext"
          rel="stylesheet" type="text/css">
    <link href="http://fonts.googleapis.com/css?family=Pacifico&subset=latin,latin-ext"
          rel="stylesheet" type="text/css">
    <link href="http://fonts.googleapis.com/css?family=Cabin+Sketch&subset=latin,latin-ext" rel="stylesheet" type="text/css">
    <link href="http://fonts.googleapis.com/css?family=Sacramento" rel="stylesheet" type="text/css">
  </head>
  <body>
    <hr/>

    <div id="table">
      <header>
        <h1 id="name">Prof. Dr. Karl Stroetmann</h1>
      </header>

      <div id="row1">
        <div class="right">
          <a id="dhbw" href="http://www.ba-stuttgart.de">Duale Hochschule Baden-W&uuml;rttemberg</a>
          <br/>Coblitzallee 1-9
          <br/>68163 Mannheim
          <br/>Germany
	  <br>
          <br/>Office: &nbsp;&nbsp;&nbsp; Raum 344B
          <br/>Phone:&nbsp;&nbsp;&nbsp; +49 621 4105-1376
          <br/>Fax:&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; +49 621 4105-1194
          <br/>Skype: &nbsp;&nbsp;&nbsp; karlstroetmann
        </div>  


        <div id="links">
          <strong class="some">Some links:</strong>
          <ul class="inlink">
            <li class="inlink">
	      My <a class="inlink" href="https://github.com/karlstroetmann?tab=repositories">lecture notes</a>,
              as well as the programs presented in class, can be found
              at <br>
              <a class="inlink" href="https://github.com/karlstroetmann?tab=repositories">https://github.com/karlstroetmann</a>.
              
            </li>
            <li class="inlink">Most of my papers can be found at <a class="inlink" href="https://www.researchgate.net/">researchgate.net</a>.</li>
            <li class="inlink">The programming language SetlX can be downloaded at <br>
              <a href="http://randoom.org/Software/SetlX"><tt class="inlink">http://randoom.org/Software/SetlX</tt></a>.
            </li>
          </ul>
        </div>
      </div>
    </div>
    
    <div id="intro">
      As I am getting old and wise, I have to accept the limits of
      my own capabilities.  I have condensed these deep philosophical
      insights into a most beautiful pearl of poetry.  I would like 
      to share these humble words of wisdom:
      
      <div class="poetry">
        I am a teacher by profession,    <br>
        mostly really by obsession;      <br>
        But even though I boldly try,    <br>
        I just cannot teach <a href="flying-pig.jpg" id="fp">pigs</a> to fly.</br>
        Instead, I slaughter them and fry.
      </div>
      
      <div class="citation">
        <div class="quote">
          Any sufficiently advanced poetry is indistinguishable from divine wisdom.
        </div>
        <div id="sign">His holiness Pope Hugo &#8555;.</div>
      </div>
    </div>
</div>

</body>
</html>
|}

In [ ]:
(* display_html is defined in ../lib/nb.ml *)

In [ ]:
display_html data

## Imports

We will use the module `Re` from the file `../lib/regex.ml`, which offers the functions of Python's module `re`
on top of the library PCRE2, to remove the 
<span style="font-variant:small-caps;">Html</span> tags and extract the text that
is embedded in the <span style="font-variant:small-caps;">Html</span> shown above.
In order to support named <span style="font-variant:small-caps;">Html5</span> entities 
we also need the table `html5`.  The OCaml standard library has no such table.  Therefore, the file
`../lib/html5.ml` contains a copy of the table `html5` from the Python module `html.entities`.  This file has
been generated by the script `../tools/gen_html5_entities.py`.  The table is a *hash table*, which is the
OCaml counterpart of a Python dictionary.

In [ ]:
load_lib "regex.ml";;
load_lib "html5.ml"

In [ ]:
Hashtbl.find html5 "uuml"

## Text Extraction Pipeline

We will approach this problem by applying a sequence of regular expression substitutions (`Re.sub`). This functional approach passes the text through a pipeline where each step systematically removes unwanted elements or decodes entities.  We begin by copying our initial <span style="font-variant:small-caps;">Html</span> string.

Since the variables of OCaml can not be changed, every step defines a *new* variable `text` that *shadows*
the previous one.

In [ ]:
let flgs = Re.[IGNORECASE; DOTALL]
let text = data

### 1. Removing `<head>` and `<script>` Elements

Once we are inside an <span style="font-variant:small-caps;">Html</span> header or inside of some
*JavaScript* code, the rules change because we want to completely discard their contents. We use the regular expression `<head>.*?</head>` to match the header block and `<script\b[^>]*>.*?</script>` to match any embedded scripts.  The part `\b[^>]*` allows the opening tag to contain attributes, as in `<script src="code.js">`, while the word boundary `\b` ensures that the name of the tag is exactly `script`. 

The variable `flgs` is the list of the two flags `IGNORECASE` and `DOTALL`.  (In Python, flags are combined
with the bitwise *or* operator `|`.)  The flag `DOTALL` ensures that the `.` character matches newlines as well, allowing us to remove multi-line blocks. The `IGNORECASE` flag handles both lowercase and uppercase tag names.

In [ ]:
let text = Re.sub ~flags:flgs {|<head>.*?</head>|}            "" text
let text = Re.sub ~flags:flgs {|<script\b[^>]*>.*?</script>|} "" text

In [ ]:
print_endline text

### 2. Removing Remaining <span style="font-variant:small-caps;">Html</span> Tags

With the `<head>` and `<script>` blocks out of the way, we can safely remove all remaining <span style="font-variant:small-caps;">Html</span> tags. The regular expression `<[^>]+>` matches any string that starts with the character `<` and ends with the character `>`. Between these two characters, there has to be a nonzero number of characters that are different from the character `>`. These matched tags are simply replaced by an empty string.

In [ ]:
let text = Re.sub {|<[^>]+>|} "" text
;;
print_endline text

### 3. Resolving Named Entities

The regular expression `&[a-zA-Z]+;?` searches for <span style="font-variant:small-caps;">Html</span> entity names. These are strings that start with the character `&` followed by the name of the entity, optionally followed by the character `;`. For example, `&auml;` is the entity name that specifies the German umlaut `ä`.

We define a replacement function `replace_named` that chops off the character `&` and uses the table `html5` to return the corresponding Unicode character.  Note that the keys of this table include the terminating semicolon, e.g. the key `"auml;"` is mapped to the string `"ä"`.  A few entities, e.g. `auml` and `nbsp`, are also contained without the semicolon, since <span style="font-variant:small-caps;">Html</span> tolerates these entities even if the semicolon is missing.  Hence, we must not chop off the semicolon.  If the entity is unknown, `Hashtbl.find_opt` returns `None` and the entity is left untouched.  The parameter `m` has the type `Re.match_`, i.e. it is a match object.

Since the replacement is a function and not a string, we use `Re.sub_fn` instead of `Re.sub`.  In Python,
`re.sub` accepts both a string and a function as its second argument.

**Function `replace_named m`**
- *Input:* `m` is a match object for a named entity like `&uuml;`.
- *Output:* The character that is denoted by the entity.  If the entity is unknown, it is returned unchanged.

In [ ]:
let replace_named (m : Re.match_) : string =
  let s = Re.group m 0 in
  match Hashtbl.find_opt html5 (String.sub s 1 (String.length s - 1)) with  (* chop off '&' *)
  | Some c -> c
  | None -> s

In [ ]:
let text = Re.sub_fn {|&[a-zA-Z]+;?|} replace_named text

In [ ]:
print_endline text

### 4. Resolving Numeric Unicode Entities

The regular expression `&#[0-9]+;?` searches for <span style="font-variant:small-caps;">Html</span> entities that specify a unicode character numerically. The corresponding strings start with the character `&` followed by the character `#` followed by digits and are optionally ended by the character `;`.

Our replacement function `replace_unicode` isolates the numeric part of the match and converts it into a
character.  `Uchar.of_int 128034` is the character `🐢`, and the function `utf8`, which is defined in the next
cell, turns a character into its UTF-8 encoding.  The function `int_of_string_opt` returns `None` if the number
is too big for an `int`, and `Uchar.is_valid` checks whether the number is a valid code point.  In Python,
both cases are handled by catching the exception `ValueError`.

**Function `replace_unicode m`**
- *Input:* `m` is a match object for a numerical entity like `&#8555;`.
- *Output:* The character whose code point is given by the entity.  If the number is not a valid code point, the entity is returned unchanged.

In [ ]:
let utf8 (c : Uchar.t) : string =
  let b = Buffer.create 4 in
  Buffer.add_utf_8_uchar b c;
  Buffer.contents b

let replace_unicode (m : Re.match_) : string =
  let s = Re.group m 0 in
  let number_str =
    if s.[String.length s - 1] = ';' then String.sub s 2 (String.length s - 3)  (* chop off '&#' and ';' *)
    else String.sub s 2 (String.length s - 2)                                   (* only chop off '&#' *)
  in
  match int_of_string_opt number_str with
  | Some n when Uchar.is_valid n -> utf8 (Uchar.of_int n)
  | _ -> s

In [ ]:
let text = Re.sub_fn {|&#[0-9]+;?|} replace_unicode text

In [ ]:
print_endline text

### 5. Cleaning Up Linebreaks

Finally, the regular expression `(\s*\n\s*)+` matches groups of whitespace containing at least one newline character. We condense these groups into a single newline character to clean up the formatting of our extracted text output.

In [ ]:
let text = Re.sub {|(\s*\n\s*)+|} "\n" text
;;
print_endline (String.trim text)